*0.4 Deep learning basics*

# GPT-2-style mini model from scratch (nanoGPT)

**The situation.** People ask "what *is* GPT, really?" The answer fits in one screen: a decoder-only transformer trained to predict the next character or token, then sampled. Andrej Karpathy's nanoGPT made that point famous. Here is the same thing, small enough to train on a laptop in a couple of minutes on real Wikipedia text.

**The recipe.** Token embedding + position embedding → N decoder blocks (causal multi-head self-attention, feed-forward, layer norms, residual connections) → a linear head over the vocabulary. Loss: cross-entropy on the next token, at every position. Generate by sampling from the softmax and appending.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Data: WikiText-2**, character level, so the vocabulary is tiny and training is fast.

In [2]:
import torch
from datasets import load_dataset

text = "".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")["train"]["text"])[
    :1_500_000
]
characters = sorted(set(text))
index = {}
for position, character in enumerate(characters):
    index[character] = position
ids = []
for character in text:
    ids.append(index[character])
data = torch.tensor(ids, dtype=torch.long)
train_data, val_data = data[:1_400_000], data[1_400_000:]
print(
    "characters of text:",
    f"{len(text):,}",
    "| vocabulary:",
    len(characters),
    "| sample:",
    repr(text[1000:1080]),
)


def batch(split, block=64, size=64):
    source = train_data if split == "train" else val_data
    starts = torch.randint(0, len(source) - block - 1, (size,))
    xs, ys = [], []
    for s in starts.tolist():
        xs.append(source[s : s + block])
        ys.append(source[s + 1 : s + block + 1])  # shifted by one
    return torch.stack(xs), torch.stack(ys)

characters of text: 1,500,000 | vocabulary: 301 | sample: 'omers . Character designer Raita Honjou and composer Hitoshi Sakimoto both retur'


**The model.** Every part is a module from the previous items; the block is written out so nothing is hidden.

In [3]:
import time

import torch.nn.functional as F
from torch import nn


def count_parameters(module):
    total = 0
    for parameter in module.parameters():
        total += parameter.numel()
    return total


class Block(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x, causal):
        h = self.norm1(x)
        x = (
            x + self.attention(h, h, h, attn_mask=causal, need_weights=False)[0]
        )  # residual: add, don't replace
        return x + self.feed_forward(self.norm2(x))


class MiniGPT(nn.Module):
    def __init__(self, vocabulary, block=64, d_model=128, heads=4, layers=4):
        super().__init__()
        self.block = block
        self.token_embedding = nn.Embedding(vocabulary, d_model)
        self.position_embedding = nn.Embedding(block, d_model)
        self.blocks = nn.ModuleList()
        for _ in range(layers):
            self.blocks.append(Block(d_model, heads))
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocabulary)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.token_embedding(ids) + self.position_embedding(positions)
        causal = nn.Transformer.generate_square_subsequent_mask(ids.shape[1])
        for block in self.blocks:
            x = block(x, causal)
        return self.head(self.norm(x))

    @torch.no_grad()
    def generate(self, ids, new_tokens, temperature=0.8):
        for _ in range(new_tokens):
            logits = (
                self(ids[:, -self.block :])[:, -1] / temperature
            )  # only the last position matters
            next_id = torch.multinomial(
                torch.softmax(logits, dim=-1), num_samples=1
            )  # sample, don't argmax
            ids = torch.cat([ids, next_id], dim=1)
        return ids


torch.manual_seed(0)
model = MiniGPT(len(characters))
print("parameters:", f"{count_parameters(model):,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

started = time.perf_counter()
for step in range(1, 1501):
    x, y = batch("train")
    loss = F.cross_entropy(model(x).reshape(-1, len(characters)), y.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step in (1, 100, 500, 1000, 1500):
        model.eval()
        with torch.no_grad():
            vx, vy = batch("val")
            val_loss = F.cross_entropy(
                model(vx).reshape(-1, len(characters)), vy.reshape(-1)
            ).item()
        model.train()
        print(f"step {step:>5}  train loss {loss.item():.2f}  val loss {val_loss:.2f}")
print(f"trained in {time.perf_counter() - started:.0f} s")
assert val_loss < 2.5

parameters: 878,893


step     1  train loss 5.95  val loss 5.43


step   100  train loss 2.55  val loss 2.48


step   500  train loss 1.92  val loss 1.92


step  1000  train loss 1.65  val loss 1.67


step  1500  train loss 1.54  val loss 1.59
trained in 112 s


**Reading the output.** Loss starts near ln(vocabulary) — uniform guessing — and falls steadily. A character-level model at loss ~1.5–2 knows spelling, spacing and common words. This is the loss curve every LLM training run shows, at a scale a laptop can run.

**Generate.** Give it a prompt and sample.

In [4]:
model.eval()
prompt_ids = []
for character in "The city of ":
    prompt_ids.append(index[character])
generated = model.generate(torch.tensor([prompt_ids]), new_tokens=200)
text_out = ""
for i in generated[0].tolist():
    text_out += characters[i]
print(text_out)

The city of the Dedificate in a creating to be compete as several open that state in the church ass the feet year 's chances . A decorated one to the rea , songs with AA rasterican Soviet AML . This commember and


**Reading the output.** English-looking text: real words, plausible punctuation, wrong facts. That is what next-character prediction on 1.4 million characters buys. GPT-2 is this architecture with 1.5 billion parameters, tokens instead of characters, and 40 GB of text; the difference is scale, not design.

```
ids ─▶ token emb + position emb ─▶ [ norm ─▶ causal MHA ─▶ +residual ─▶ norm ─▶ FF ─▶ +residual ] × 4 ─▶ norm ─▶ head ─▶ next-char logits
generate: sample from softmax(logits / temperature) ─▶ append ─▶ repeat
```

**The rule to remember.** GPT = decoder blocks + next-token loss + sampling. Everything else in LLM training — scale, data, RLHF — sits on top of this exact loop.

| Use it when | Don't when | Instead use |
|---|---|---|
| understanding LLMs end to end; small domain models for research | any production text generation | a pretrained model (Layer 1 onward) |

**Watch out**
- Pre-norm (norm before attention) and residual connections are what let deep stacks train; remove either and loss stalls.
- `temperature` here is the 0.2 sampling item in action; try 0.3 and 1.5 to see repetition vs noise.
- Character models learn spelling first and meaning much later; the real thing uses BPE tokens (0.3) to skip the spelling stage.